# 04 — Backtest: HoldCash (engine correctness check)

HoldCashStrategy must end with final_equity == starting_capital and zero trades.

In [1]:
# Notebook is location-agnostic: walk up to find the project root
# (the dir containing pyproject.toml) and chdir there before any imports.
import os, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "pyproject.toml").exists():
    _p = _p.parent
if (_p / "pyproject.toml").exists():
    os.chdir(_p)
del _p


In [2]:
from datetime import datetime, timezone
from src.layer1_research.backtesting.config import BacktestConfig
from src.layer1_research.backtesting.runner import BacktestRunner
from src.layer1_research.backtesting.strategies.examples.hold_cash import HoldCashStrategy
from src.layer1_research.backtesting.reporting.cli_report import print_report

In [3]:
config = BacktestConfig(
    catalog_path="data/catalog",
    start=datetime(2024, 6, 1, tzinfo=timezone.utc),
    end=datetime(2024, 9, 1, tzinfo=timezone.utc),
    strategy_name="hold_cash",
    starting_capital=10_000.0,
    data_mode="trade",
)

In [4]:
result = BacktestRunner(config).run(HoldCashStrategy)
print_report(result.metrics())

2024-08-31T23:59:49.000000000Z [WARN] BACKTESTER-001.HoldCashStrategy: The `Strategy.on_stop` handler was called when not overridden. It's expected that any actions required when stopping the strategy occur here, such as unsubscribing from data



  Backtest results
  Total Return:                0.00%
  Sharpe Ratio:                 nan
  Sortino Ratio:                nan
  Max Drawdown:                0.00%
  Calmar Ratio:                0.00
  Total Trades:                   0
  Win Rate:                    0.0%
  Avg Win:             $       0.00
  Avg Loss:            $       0.00
  Profit Factor:               0.00
  Total Fees:          $       0.00
  Fee Drag:                    0.0%
  Avg Slippage:               0.0 bps
  Avg Edge @ Order:          0.0000
  Edge Realization:            0.00



### Invariants

In [5]:
final_eq = float(result.equity_curve.iloc[-1])
# Allow micro-cent float drift; engine should not move capital meaningfully.
assert abs(final_eq - 10_000.0) < 1e-2, f"Expected ~10000, got {final_eq}"
assert len(result.trades) == 0
assert len(result.signals) == 0
print("HoldCash invariants hold.")


HoldCash invariants hold.
